In [2]:
import numpy
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms

In [3]:
transform = transforms.ToTensor()

In [22]:
train_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [23]:
test_data = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [24]:
# Converting the Torch Tensor to Numpy Array
X_train = train_data.data.numpy().astype(np.float32)
y_train = train_data.targets.numpy()

In [25]:
X_train.shape

(60000, 28, 28)

In [26]:
X_test = test_data.data.numpy().astype(np.float32)
y_test = test_data.targets.numpy()

In [27]:
# Normalizing the dataset to range btw 0 to 1
X_train = X_train / 255.0
X_test = X_test / 255.0

In [28]:
print(X_train.shape)

(60000, 28, 28)


In [29]:
X_train[:3]

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], shape=(3, 28, 28), dtype=float32)

In [30]:
X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

In [31]:
print(X_train.shape)
print(X_test.shape)

(60000, 784)
(10000, 784)


In [11]:
X_train[:3]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(3, 784), dtype=float32)

In [12]:
# Initialize the weights and bias
np.random.seed(42)

In [13]:
W1 = np.random.randn(784, 128) * np.sqrt(2.0 / 784)
b1 = np.zeros((1, 128))

In [14]:
W2 = np.random.randn(128, 10) * np.sqrt(2.0 / 128)
b2 = np.zeros((1, 10))

In [15]:
print("W1:", W1.shape)
print("b1:", b1.shape)
print("W2:", W2.shape)
print("b2:", b2.shape)

W1: (784, 128)
b1: (1, 128)
W2: (128, 10)
b2: (1, 10)


In [36]:
def forward_propagation(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = np.maximum(0, Z1)

    Z2 = A1 @ W2 + b2

    return Z2


In [39]:
predictions = forward_propagation(X_train[:1], W1, b1, W2, b2)

In [40]:
predictions

array([[ 0.45827062,  0.01013389,  0.06027466, -0.71404021, -0.27256249,
         0.36652482, -0.49724009,  0.48245591, -0.15971654,  0.40063565]])

In [41]:
predicted_digit = np.argmax(predictions, axis=1)
print(predicted_digit)

[7]


In [42]:
def compute_loss(scores, y):
    scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    probabilities = exp_scores / np.sum(
        exp_scores, axis=1, keepdims=True
    )

    # Get the probability assigned to the correct digit
    correct_probabilities = probabilities[
        np.arange(len(y)), y
    ]

    # Calculate average cross-entropy loss
    loss = -np.mean(np.log(correct_probabilities))

    return loss

In [43]:
scores = forward_propagation(
    X_train[:5], W1, b1, W2, b2
)

loss = compute_loss(scores, y_train[:5])

print("True digits:", y_train[:5])
print("Loss:", loss)

True digits: [5 0 4 1 9]
Loss: 2.3268776163629936


In [49]:
def output_gradients(Z2, y):
    # Convert scores into probabilities
    Z2 = Z2 - np.max(Z2, axis=1, keepdims=True)
    exp_scores = np.exp(Z2)
    P = exp_scores / np.sum(
        exp_scores, axis=1, keepdims=True
    )

    # Copy probabilities so we can modify them
    dZ2 = P.copy()

    # Subtract 1 at each correct digit's position
    dZ2[np.arange(len(y)), y] -= 1

    # Average gradients across the batch
    dZ2 /= len(y)

    return dZ2

In [50]:
scores = forward_propagation(
    X_train[:5], W1, b1, W2, b2
)

dZ2 = output_gradients(scores, y_train[:5])

print("Gradient shape:", dZ2.shape)
print(dZ2)

Gradient shape: (5, 10)
[[ 0.02895992  0.0185001   0.01945136  0.00896746  0.01394442 -0.17357879
   0.01113844  0.02966886  0.01561022  0.02733801]
 [-0.18558636  0.02365323  0.02297634  0.00777572  0.01000639  0.04035423
   0.02029468  0.02772553  0.01673824  0.016062  ]
 [ 0.01936469  0.01994665  0.01578149  0.01187847 -0.18451989  0.01956953
   0.0176669   0.03327244  0.01958276  0.02745697]
 [ 0.01906358 -0.18061954  0.02296832  0.01379905  0.02168151  0.0214184
   0.01304608  0.01904207  0.02264268  0.02695785]
 [ 0.01698286  0.02153596  0.0165929   0.01669177  0.01120136  0.01770618
   0.02194037  0.03191222  0.02063164 -0.17519525]]


In [51]:
def output_layer_gradients(A1, dZ2):
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    return dW2, db2

In [52]:
X_batch = X_train[:5]
y_batch = y_train[:5]

Z2 = forward_propagation(X_batch, W1, b1, W2, b2)
dZ2 = output_gradients(Z2, y_batch)

# We need A1 from the hidden layer
Z1 = X_batch @ W1 + b1
A1 = np.maximum(0, Z1)

dW2, db2 = output_layer_gradients(A1, dZ2)

print("dW2 shape:", dW2.shape)
print("db2 shape:", db2.shape)

dW2 shape: (128, 10)
db2 shape: (1, 10)


In [53]:
def hidden_layer_gradients(X, Z1, dA1):
    # Pass gradients through ReLU
    dZ1 = dA1 * (Z1 > 0)

    # Calculate gradients for first-layer weights and biases
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return dW1, db1

In [54]:
X_batch = X_train[:5]
y_batch = y_train[:5]

# Forward propagation
Z1 = X_batch @ W1 + b1
A1 = np.maximum(0, Z1)
Z2 = A1 @ W2 + b2

# Output gradients
dZ2 = output_gradients(Z2, y_batch)
dW2, db2 = output_layer_gradients(A1, dZ2)

# Send gradients backward to the hidden layer
dA1 = dZ2 @ W2.T
dW1, db1 = hidden_layer_gradients(X_batch, Z1, dA1)

print("dW1:", dW1.shape)
print("db1:", db1.shape)
print("dW2:", dW2.shape)
print("db2:", db2.shape)

dW1: (784, 128)
db1: (1, 128)
dW2: (128, 10)
db2: (1, 10)


In [55]:
def update_parameters(
    W1, b1, W2, b2,
    dW1, db1, dW2, db2,
    learning_rate
):
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    return W1, b1, W2, b2

In [56]:
learning_rate = 0.01

W1, b1, W2, b2 = update_parameters(
    W1, b1, W2, b2,
    dW1, db1, dW2, db2,
    learning_rate
)

print("Weights and biases updated!")

Weights and biases updated!


In [57]:
def train(X_train, y_train, epochs=5, batch_size=64, learning_rate=0.01):
    global W1, b1, W2, b2

    n_samples = X_train.shape[0]

    for epoch in range(epochs):
        # Shuffle the training examples
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        total_loss = 0.0

        for start in range(0, n_samples, batch_size):
            end = start + batch_size

            X_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]

            # Forward propagation
            Z1 = X_batch @ W1 + b1
            A1 = np.maximum(0, Z1)
            Z2 = A1 @ W2 + b2

            # Calculate loss
            loss = compute_loss(Z2, y_batch)
            total_loss += loss * len(y_batch)

            # Backpropagation
            dZ2 = output_gradients(Z2, y_batch)
            dW2, db2 = output_layer_gradients(A1, dZ2)

            dA1 = dZ2 @ W2.T
            dW1, db1 = hidden_layer_gradients(X_batch, Z1, dA1)

            # Update parameters
            W1, b1, W2, b2 = update_parameters(
                W1, b1, W2, b2,
                dW1, db1, dW2, db2,
                learning_rate
            )

        average_loss = total_loss / n_samples
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {average_loss:.4f}")

In [58]:
train(X_train, y_train, epochs=5, batch_size=64, learning_rate=0.01)

Epoch 1/5, Loss: 0.8819
Epoch 2/5, Loss: 0.4175
Epoch 3/5, Loss: 0.3517
Epoch 4/5, Loss: 0.3184
Epoch 5/5, Loss: 0.2956


In [59]:
def evaluate(X_test, y_test, batch_size=64):
    total_correct = 0
    total_loss = 0.0
    total_samples = len(y_test)

    for start in range(0, total_samples, batch_size):
        end = start + batch_size

        X_batch = X_test[start:end]
        y_batch = y_test[start:end]

        # Forward propagation
        scores = forward_propagation(
            X_batch, W1, b1, W2, b2
        )

        # Calculate loss
        loss = compute_loss(scores, y_batch)
        total_loss += loss * len(y_batch)

        # Select the digit with the highest score
        predictions = np.argmax(scores, axis=1)

        # Count correct predictions
        total_correct += np.sum(predictions == y_batch)

    accuracy = total_correct / total_samples * 100
    average_loss = total_loss / total_samples
    incorrect = total_samples - total_correct

    print(f"Test images: {total_samples}")
    print(f"Correct predictions: {total_correct}")
    print(f"Incorrect predictions: {incorrect}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Average loss: {average_loss:.4f}")

    return accuracy

In [61]:
test_accuracy = evaluate(X_train, y_train)

Test images: 60000
Correct predictions: 55210
Incorrect predictions: 4790
Accuracy: 92.02%
Average loss: 0.2845
